# DWD-Tagesmitteltemperatur prognostizieren

Dieses Notebook sagt die Tagesmitteltemperatur für Berlin-Dahlem einen Tag voraus. Als Eingaben dienen die Temperaturen der fünf vorangegangenen Tage und der Kalendertag (1–365). Die vorbereitete Trainingsdatei wird aus den DWD-Tageswerten geladen.

Es wird ein Random-Forest-Regressor verwendet. Der Test erfolgt chronologisch auf 2025; die Testtemperaturen werden beim Training nicht verwendet.

## Trainingsdaten laden

Die fünf Temperatur-Lags heißen `temp_t_minus_5_c` bis `temp_t_minus_1_c`; `tag_im_jahr_1_365` beschreibt den Zieltag. `ziel_temp_durch_c` ist die zu prognostizierende Tagesmitteltemperatur. Schaltjahre sind auf 365 Tage abgebildet; der 29. Februar wird Tag 59 zugeordnet.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

trainingsdatei = Path("DWD_Temperaturprognose_Trainingsdaten.csv")
datensatz = pd.read_csv(trainingsdatei, parse_dates=["datum"])

feature_spalten = [
    "temp_t_minus_5_c",
    "temp_t_minus_4_c",
    "temp_t_minus_3_c",
    "temp_t_minus_2_c",
    "temp_t_minus_1_c",
    "tag_im_jahr_1_365",
]
ziel_spalte = "ziel_temp_durch_c"

print(f"Samples: {len(datensatz)}")
print(f"Zeitraum: {datensatz['datum'].min():%d.%m.%Y} bis {datensatz['datum'].max():%d.%m.%Y}")
display(datensatz.head())

## Trainieren und auf 2025 testen

Das Modell wird nur mit Daten bis einschließlich 2024 trainiert. Die Werte von 2025 bilden den unangetasteten Testzeitraum. Für jeden Testtag sind die fünf vorherigen tatsächlichen Messwerte bekannt, wie bei einer fortlaufenden Ein-Tages-Prognose. Als Referenz dient die Persistenz-Baseline: Die Temperatur von morgen entspricht der von heute.

In [ ]:
train = datensatz[datensatz["datum"] < "2025-01-01"]
test = datensatz[datensatz["datum"].dt.year == 2025].copy()

modell = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
)
modell.fit(train[feature_spalten], train[ziel_spalte])

test["prognose_rf_c"] = modell.predict(test[feature_spalten])
test["prognose_vortag_c"] = test["temp_t_minus_1_c"]

metriken = []
for name, vorhersage in [
    ("Random Forest", test["prognose_rf_c"]),
    ("Vortag-Baseline", test["prognose_vortag_c"]),
]:
    metriken.append({
        "Modell": name,
        "Testtage": len(test),
        "MAE (°C)": mean_absolute_error(test[ziel_spalte], vorhersage),
        "RMSE (°C)": mean_squared_error(test[ziel_spalte], vorhersage) ** 0.5,
        "R²": r2_score(test[ziel_spalte], vorhersage),
    })

display(pd.DataFrame(metriken).set_index("Modell").round(3))

## Prognose und Messung im Testjahr

Das Diagramm zeigt, wie nah die Modellprognosen im Testjahr an den gemessenen Tagesmitteln liegen.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(test["datum"], test[ziel_spalte], label="DWD-Messung", color="black", linewidth=1.1)
ax.plot(test["datum"], test["prognose_rf_c"], label="Random Forest", color="royalblue", alpha=0.85)
ax.plot(test["datum"], test["prognose_vortag_c"], label="Vortag-Baseline", color="darkorange", alpha=0.6, linewidth=0.9)
ax.set_title("Ein-Tages-Prognose der Tagesmitteltemperatur: Testjahr 2025")
ax.set_xlabel("Datum")
ax.set_ylabel("Temperatur (°C)")
ax.grid(True, alpha=0.25)
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

## Prognose für den nächsten Tag

Nach der unabhängigen Bewertung auf 2025 wird ein neues Modell mit allen verfügbaren Trainingssamples trainiert. Es verwendet die fünf letzten DWD-Tagesmittel und prognostiziert den Kalendertag nach dem letzten Datenbankeintrag. Das ist eine einfache statistische Prognose, keine Wettervorhersage mit Luftdruck- oder Modellwetterdaten.

In [ ]:
finales_modell = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
)
finales_modell.fit(datensatz[feature_spalten], datensatz[ziel_spalte])

wetterdatei = Path("DWD_Berlin_Dahlem_Tagesdaten_2020_heute.csv")
wetter_roh = pd.read_csv(wetterdatei, parse_dates=["datum"]).dropna(subset=["temp_durch_c"])
letzter_tag = wetter_roh["datum"].max()
letzte_fuenf_tage = pd.date_range(end=letzter_tag, periods=5, freq="D")
letzte_temperaturen = wetter_roh.set_index("datum")["temp_durch_c"].reindex(letzte_fuenf_tage)
if letzte_temperaturen.isna().any():
    raise ValueError("Für die fünf letzten Kalendertage liegen keine vollständigen Messwerte vor.")

prognose_datum = letzter_tag + pd.Timedelta(days=1)
tag_im_jahr = prognose_datum.dayofyear
if prognose_datum.strftime("%m-%d") == "02-29":
    tag_im_jahr = 59
elif prognose_datum.is_leap_year and prognose_datum.month > 2:
    tag_im_jahr -= 1

neue_eingabe = pd.DataFrame([
    [*letzte_temperaturen.to_list(), tag_im_jahr]
], columns=feature_spalten)
prognose = finales_modell.predict(neue_eingabe)[0]
print(f"Prognose für {prognose_datum:%d.%m.%Y}: {prognose:.1f} °C Tagesmitteltemperatur")